# Week 6 — **Budget‑Aware RFT for Product Pricer** (auto‑fallback to SFT)

This notebook eliminates the **RFT file‑format** issue *and* prevents the
**quota hard‑limit** error:

> *Creating this reinforcement fine‑tuning job would potentially exceed your hard limit … We require at least 4 hours of training quota (~USD 400 minimum).*

### What this notebook does
1. **Builds valid RFT data** (no `system` messages in `messages`, *user‑only*), or converts your SFT JSONL to RFT JSONL.
2. **Checks budget**. If your available budget is below a configurable threshold (defaults to USD 400 for RFT), it **automatically switches to SFT** on `gpt-4o-mini` so the job will run under a few dollars.
3. Provides identical **evaluation** helpers for the resulting fine‑tuned model.

> **Why auto‑fallback?** Current OpenAI RFT requires a high *minimum* training quota to start any job. If you only have a few dollars of quota, RFT will fail at creation time. This notebook avoids that failure by switching to an SFT job that fits your budget.

## 0) Install (optional)

In [1]:
# !pip -q install --upgrade openai wandb python-dotenv matplotlib numpy pandas tqdm requests


## 1) Environment & budget guard

- Set `OPENAI_API_KEY` in your environment or `.env`
- Optionally set `WANDB_API_KEY` if you want W&B logging
- **Budget variables**:
  - `AVAILABLE_BUDGET_USD` (your remaining quota; default: `2.95` for safety)
  - `RFT_MIN_START_USD`    (minimum required to start RFT; default: `400.0`)

In [2]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# --- API keys ---
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
USE_WANDB = bool(os.getenv("WANDB_API_KEY"))

if USE_WANDB:
    import wandb
    wandb.login()

# --- Budget guard ---
def _to_float(s, default):
    try:
        return float(s)
    except Exception:
        return float(default)

AVAILABLE_BUDGET_USD = _to_float(os.getenv("AVAILABLE_BUDGET_USD", "2.95"), 2.95)
RFT_MIN_START_USD    = _to_float(os.getenv("RFT_MIN_START_USD", "400.0"), 400.0)

AUTO_MODE = "RFT" if AVAILABLE_BUDGET_USD >= RFT_MIN_START_USD else "SFT"

assert os.getenv("OPENAI_API_KEY") and os.getenv("OPENAI_API_KEY") != 'your-key-if-not-using-env', \
    "OPENAI_API_KEY is required. Please set it in a .env file or environment."

print(f"Environment ready. AVAILABLE_BUDGET_USD={AVAILABLE_BUDGET_USD:.2f} | RFT_MIN_START_USD={RFT_MIN_START_USD:.2f}")
print(f"Selected training mode: {AUTO_MODE} (auto-switched based on budget).")


wandb: Currently logged in as: hafnium (hafnium49) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Environment ready. AVAILABLE_BUDGET_USD=2.95 | RFT_MIN_START_USD=400.00
Selected training mode: SFT (auto-switched based on budget).


## 2) Imports

In [3]:
import os, re, json, pickle, math, random
from pathlib import Path
from typing import List, Dict, Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from openai import OpenAI
client = OpenAI()

print("OpenAI SDK ready.")


OpenAI SDK ready.


## 3) Locate data

Use either **SFT JSONL** (previous runs) **or** `train.pkl`/`test.pkl`.
If nothing is found, please upload them to the working directory.

In [4]:
SFT_TRAIN_JSONL = Path("fine_tune_train.jsonl")          # optional, if you already have SFT data
SFT_VAL_JSONL   = Path("fine_tune_validation.jsonl")     # optional, if you already have SFT data

DATA_DIR = Path(".")
TRAIN_PKL = DATA_DIR / "train.pkl"
TEST_PKL  = DATA_DIR / "test.pkl"

has_sft = SFT_TRAIN_JSONL.exists() and SFT_VAL_JSONL.exists()
has_pkl = TRAIN_PKL.exists() and TEST_PKL.exists()

print(f"SFT JSONL present: {has_sft}")
print(f"PKL present:       {has_pkl}")

if not has_sft and not has_pkl:
    print("⚠️ No SFT JSONL or PKL datasets found in current directory.")
    print("   Re-upload: train.pkl/test.pkl or fine_tune_*.jsonl")


SFT JSONL present: True
PKL present:       True


### 3.1 Optional helpers (`items.Item`, `testing.Tester`)

In [5]:
Item = None
Tester = None
try:
    from items import Item
    from testing import Tester
    print("Imported items.Item and testing.Tester.")
except Exception as e:
    print("Using fallback Tester:", e)
    class _FallbackItem:
        def __init__(self, text, price):
            self.text = text
            self.price = float(price)
        def test_prompt(self):
            return f"How much does this cost?\n\n{self.text}\n\nPrice is $"
    class _FallbackTester:
        @staticmethod
        def test(fn, dataset, max_n=None, name="model"):
            maes = []
            n = len(dataset) if max_n is None else min(max_n, len(dataset))
            for i in range(n):
                guess, truth = fn(dataset[i]), float(dataset[i].price)
                err = abs(guess - truth); maes.append(err)
                color = "\x1b[92m" if err <= 50 else ("\x1b[93m" if err <= 150 else "\x1b[91m")
                print(f"{color}{i+1}: Guess: ${guess:.2f} Truth: ${truth:.2f} Error: ${err:.2f}\x1b[0m")
            print(f"\n{name} — MAE: ${np.mean(maes):.2f} | Median: ${np.median(maes):.2f}")
    Tester = _FallbackTester


Imported items.Item and testing.Tester.


## 4) Utilities

In [6]:
def get_price(s: str) -> float:
    s = (s or "").replace("$","").replace(",","")
    m = re.search(r"[-+]?\d*\.?\d+", s)
    return float(m.group()) if m else 0.0


## 5) Build **RFT JSONL** (no `system` messages) and **SFT JSONL** (tiny, cheap)
We generate both formats. The training path chosen later depends on your budget.

- RFT messages: **user‑only**; `reference_answer` holds the ground truth string.
- SFT messages: system/user/assistant triplets, sized to be **very small** (≤ 100 examples) so it will run with a few dollars of quota.

In [7]:
def item_to_rft_row(item):
    instruction = "Estimate the price in USD. Output only a number (no $ or text)."
    user = item.test_prompt().replace(" to the nearest dollar","").replace("\n\nPrice is $","")
    return {
        "messages": [{"role":"user","content": instruction + "\n\n" + user}],
        "reference_answer": f"{float(item.price):.2f}"
    }

def item_to_sft_row(item):
    system = "You estimate prices of items. Reply only with the price, no explanation."
    user = item.test_prompt().replace(" to the nearest dollar","").replace("\n\nPrice is $","")
    assistant = f"Price is ${float(item.price):.2f}"
    return {"messages":[
        {"role":"system","content": system},
        {"role":"user","content":   user},
        {"role":"assistant","content": assistant}
    ]}

def write_jsonl(rows, path: Path):
    n = 0
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
            n += 1
    print(f"Wrote {n} rows -> {path}")


In [8]:
# Prepare selections
random.seed(42); np.random.seed(42)

OUT = Path("week6_budget_safe"); OUT.mkdir(exist_ok=True, parents=True)
RFT_TRAIN = OUT / "rft_train.jsonl"
RFT_VAL   = OUT / "rft_val.jsonl"
SFT_TRAIN = OUT / "sft_train_small.jsonl"
SFT_VAL   = OUT / "sft_val_small.jsonl"

# Tiny defaults for SFT (cheap). Adjust upward if you add budget.
SFT_N_TRAIN = int(os.getenv("SFT_N_TRAIN", "80"))
SFT_N_VAL   = int(os.getenv("SFT_N_VAL", "20"))

# Modest defaults for RFT data (won't be used if budget too low).
RFT_N_TRAIN = int(os.getenv("RFT_N_TRAIN", "500"))
RFT_N_VAL   = int(os.getenv("RFT_N_VAL", "50"))

# Load items from either PKL or skip if we only have SFT JSONL
items_loaded = False
if has_pkl:
    with open(TRAIN_PKL, "rb") as f: train_items = pickle.load(f)
    with open(TEST_PKL, "rb") as f:  test_items  = pickle.load(f)
    if len(train_items)>0 and not hasattr(train_items[0], "test_prompt"):
        class _Item:
            def __init__(self, d):
                self.text  = d["text"]; self.price = float(d["price"])
            def test_prompt(self):
                return f"How much does this cost?\n\n{self.text}\n\nPrice is $"
        train_items = [ _Item(d) for d in train_items ]
        test_items  = [ _Item(d) for d in test_items ]
    items_loaded = True

if items_loaded:
    idx = np.random.permutation(len(train_items))
    # RFT
    rft_fit = [train_items[i] for i in idx[:min(RFT_N_TRAIN, len(idx))]]
    rft_val = [train_items[i] for i in idx[min(RFT_N_TRAIN, len(idx)):min(RFT_N_TRAIN+RFT_N_VAL, len(idx))]]
    write_jsonl([item_to_rft_row(it) for it in rft_fit], RFT_TRAIN)
    write_jsonl([item_to_rft_row(it) for it in rft_val], RFT_VAL)
    # SFT (tiny)
    sft_fit = [train_items[i] for i in idx[:min(SFT_N_TRAIN, len(idx))]]
    sft_val = [train_items[i] for i in idx[min(SFT_N_TRAIN, len(idx)):min(SFT_N_TRAIN+SFT_N_VAL, len(idx))]]
    write_jsonl([item_to_sft_row(it) for it in sft_fit], SFT_TRAIN)
    write_jsonl([item_to_sft_row(it) for it in sft_val], SFT_VAL)
else:
    # If only SFT JSONL exists, we still can launch SFT; for RFT, try to convert
    # a small slice from SFT to RFT-compatible rows (strip system/assistant).
    def _extract_user_messages(obj):
        parts = []
        for m in obj.get("messages", []):
            if m.get("role") == "user":
                parts.append(m.get("content", ""))
        return "\n".join(parts).strip()

    def _extract_ref(obj):
        # Use last assistant message if present
        for m in reversed(obj.get("messages", [])):
            if m.get("role") == "assistant":
                price = get_price(m.get("content", ""))
                if price:
                    return f"{price:.2f}"
        return ""

    # Convert a small slice for SFT (copy as-is) and RFT (user-only + reference_answer)
    sft_rows = []
    with SFT_TRAIN_JSONL.open("r", encoding="utf-8") as fin:
        for i, line in enumerate(fin):
            if i >= 100: break
            try:
                obj = json.loads(line)
                sft_rows.append(obj)
            except Exception:
                continue
    write_jsonl(sft_rows[:80], SFT_TRAIN)
    write_jsonl(sft_rows[80:100], SFT_VAL)

    rft_rows = []
    with SFT_TRAIN_JSONL.open("r", encoding="utf-8") as fin:
        for i, line in enumerate(fin):
            if i >= 600: break
            try:
                obj = json.loads(line)
            except Exception:
                continue
            user = _extract_user_messages(obj)
            ref  = _extract_ref(obj)
            if not (user and ref): 
                continue
            instruction = "Estimate the price in USD. Output only a number (no $ or text)."
            rft_rows.append({
                "messages": [{"role":"user","content": instruction + "\n\n" + user.replace("\n\nPrice is $","")}],
                "reference_answer": ref
            })
    write_jsonl(rft_rows[:500], RFT_TRAIN)
    write_jsonl(rft_rows[500:550], RFT_VAL)


Wrote 500 rows -> week6_budget_safe/rft_train.jsonl
Wrote 50 rows -> week6_budget_safe/rft_val.jsonl
Wrote 80 rows -> week6_budget_safe/sft_train_small.jsonl
Wrote 20 rows -> week6_budget_safe/sft_val_small.jsonl


## 6) Python grader for RFT (numeric reward)
(Only used if `AUTO_MODE == "RFT"`)

In [9]:
import inspect, math, re, requests

def price_grader(sample, item):
    '''
    sample: {"output_text": "..."}  # model raw output
    item:   {"reference_answer": "123.45"}
    return: float in [0,1]
    '''
    def parse_num(s):
        if s is None: return None
        s = s.replace(",", "")
        m = re.search(r"[-+]?\d*\.?\d+", s)
        return float(m.group()) if m else None

    pred  = parse_num(sample.get("output_text", ""))
    truth = parse_num(item.get("reference_answer", ""))

    if pred is None or truth is None:
        return 0.0

    scale = max(25.0, 0.20 * max(1.0, truth))
    score = math.exp(-abs(pred - truth) / scale)

    if re.search(r"[^\d\.\-+]", (sample.get("output_text","").strip())):
        score *= 0.98

    return float(max(0.0, min(1.0, score)))

def build_python_grader_payload(grader_fn):
    src = inspect.getsource(grader_fn)
    if not src.strip().startswith("def grade("):
        src = src.replace(grader_fn.__name__, "grade", 1)
    return {"type":"python", "source": src}

GRADER_PAYLOAD = build_python_grader_payload(price_grader)
print("Grader ready (for RFT path).")


Grader ready (for RFT path).


### 6.1 Optional: validate grader (skip if endpoint is unavailable)

In [10]:
try:
    headers = {"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"}
    resp = requests.post(
        "https://api.openai.com/v1/fine_tuning/alpha/graders/validate",
        json={"grader": GRADER_PAYLOAD},
        headers=headers, timeout=30
    )
    print("Grader validation:", resp.status_code, resp.text[:120])
except Exception as e:
    print("Grader validation skipped:", e)


Grader validation: 200 {
  "grader": {
    "type": "python",
    "source": "def grade(sample, item):\n    '''\n    sample: {\"output_text\": \"


## 7) Launch **budget‑aware** fine‑tuning

- If `AUTO_MODE == "RFT"` → launch **RFT** on `o4-mini-2025-04-16`  
- Else → launch **SFT** on `gpt-4o-mini-2024-07-18` (cheap)

In [11]:
from datetime import datetime

BASE_RFT = "o4-mini-2025-04-16"          # reasoning model (RFT only)
BASE_SFT = "gpt-4o-mini-2024-07-18"      # cheap, good baseline

EPOCHS_RFT = int(os.getenv("EPOCHS_RFT", "1"))   # keep tiny if you do have quota
EPOCHS_SFT = int(os.getenv("EPOCHS_SFT", "1"))   # 1 epoch keeps cost low

REASONING  = os.getenv("REASONING_EFFORT", "low")  # none/low/medium/high

# Upload whichever files we'll need
def _upload(path: Path):
    with path.open("rb") as f:
        return client.files.create(file=f, purpose="fine-tune")

if AUTO_MODE == "RFT":
    train_file = _upload(RFT_TRAIN)
    val_file   = _upload(RFT_VAL)
    print("Uploaded (RFT):", train_file.id, val_file.id)

    integrations = []
    if USE_WANDB:
        integrations = [{"type":"wandb", "wandb": {"project":"gpt-pricer-rft"}}]

    job = client.fine_tuning.jobs.create(
        training_file=train_file.id,
        validation_file=val_file.id,
        model=BASE_RFT,
        seed=42,
        suffix=f"pricer-rft-budget",
        method={
            "type":"reinforcement",
            "reinforcement":{
                "grader": GRADER_PAYLOAD,
                "hyperparameters":{
                    "n_epochs": int(EPOCHS_RFT),
                    "eval_interval": 20,
                    "eval_samples": 2,
                    "compute_multiplier": 0.25,         # shrink compute/time
                    "reasoning_effort": REASONING
                }
            }
        },
        integrations=integrations
    )
    MODE_USED = "RFT"
else:
    train_file = _upload(SFT_TRAIN)
    val_file   = _upload(SFT_VAL)
    print("Uploaded (SFT):", train_file.id, val_file.id)

    integrations = []
    if USE_WANDB:
        integrations = [{"type":"wandb", "wandb": {"project":"gpt-pricer-sft"}}]

    job = client.fine_tuning.jobs.create(
        training_file=train_file.id,
        validation_file=val_file.id,
        model=BASE_SFT,
        seed=42,
        suffix=f"pricer-sft-budget",
        hyperparameters={"n_epochs": int(EPOCHS_SFT)},
        integrations=integrations
    )
    MODE_USED = "SFT"

print(f"{MODE_USED} job:", job.id, "| status:", job.status)


Uploaded (SFT): file-E6fWQ9Fg3dD8Kc5W9uWq3j file-6NFKU2WULqnjUdQKiFnaK9
SFT job: ftjob-VjlkgmpRAW4YQDONXn46zBLu | status: validating_files


In [ ]:
# Uploaded (SFT): file-E6fWQ9Fg3dD8Kc5W9uWq3j file-6NFKU2WULqnjUdQKiFnaK9
# SFT job: ftjob-VjlkgmpRAW4YQDONXn46zBLu | status: validating_files

## 8) Monitor job

In [16]:
def show_status(job_id, limit_events=10):
    j = client.fine_tuning.jobs.retrieve(job_id)
    print("Job:", j.id, "| status:", j.status, "| base:", j.model, "| fine_tuned:", j.fine_tuned_model)
    ev = client.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=limit_events)
    for e in ev.data:
        print(f"- [{e.created_at}] {e.level}: {e.message}")
    return j

_ = show_status(job.id)


Job: ftjob-VjlkgmpRAW4YQDONXn46zBLu | status: succeeded | base: gpt-4o-mini-2024-07-18 | fine_tuned: ft:gpt-4o-mini-2024-07-18:personal:pricer-sft-budget:CLEzq8mW
- [1759179342] info: The job has successfully completed
- [1759179337] info: Usage policy evaluations completed, model is now enabled for sampling
- [1759179337] info: Moderation checks for snapshot ft:gpt-4o-mini-2024-07-18:personal:pricer-sft-budget:CLEzq8mW passed.
- [1759178570] info: Evaluating model against our usage policies
- [1759178570] info: New fine-tuned model created
- [1759178517] info: Step 80/80: training loss=0.56, validation loss=0.68, full validation loss=1.18
- [1759178511] info: Step 79/80: training loss=1.91, validation loss=1.35
- [1759178505] info: Step 78/80: training loss=0.71, validation loss=1.25
- [1759178502] info: Step 77/80: training loss=0.87, validation loss=0.70
- [1759178496] info: Step 76/80: training loss=1.30, validation loss=1.14


In [ ]:
# Job: ftjob-VjlkgmpRAW4YQDONXn46zBLu | status: succeeded | base: gpt-4o-mini-2024-07-18 | fine_tuned: ft:gpt-4o-mini-2024-07-18:personal:pricer-sft-budget:CLEzq8mW
# - [1759179342] info: The job has successfully completed
# - [1759179337] info: Usage policy evaluations completed, model is now enabled for sampling
# - [1759179337] info: Moderation checks for snapshot ft:gpt-4o-mini-2024-07-18:personal:pricer-sft-budget:CLEzq8mW passed.
# - [1759178570] info: Evaluating model against our usage policies
# - [1759178570] info: New fine-tuned model created
# - [1759178517] info: Step 80/80: training loss=0.56, validation loss=0.68, full validation loss=1.18
# - [1759178511] info: Step 79/80: training loss=1.91, validation loss=1.35
# - [1759178505] info: Step 78/80: training loss=0.71, validation loss=1.25
# - [1759178502] info: Step 77/80: training loss=0.87, validation loss=0.70
# - [1759178496] info: Step 76/80: training loss=1.30, validation loss=1.14

## 9) Inference & evaluation

In [21]:
def infer_messages(item, mode):
    if mode == "RFT":
        instruction = "Estimate the price in USD. Output only a number (no $ or text)."
        user = item.test_prompt().replace(" to the nearest dollar","").replace("\n\nPrice is $","")
        return [{"role":"user","content": instruction + "\n\n" + user}]
    else:
        system = "You estimate prices of items. Reply only with the price, no explanation."
        user = item.test_prompt().replace(" to the nearest dollar","").replace("\n\nPrice is $","")
        return [
            {"role":"system","content": system},
            {"role":"user","content": user}
        ]

def predict_price(item, model_name, mode):
    rsp = client.chat.completions.create(
        model=model_name,
        messages=infer_messages(item, mode),
        temperature=0, max_tokens=8, seed=42
    )
    return get_price(rsp.choices[0].message.content)

def evaluate_predictor(predict_fn, dataset, max_n=None, name="model"):
    """Evaluate a predictor function on a dataset with optional limit."""
    from typing import Callable
    maes = []
    n = len(dataset) if max_n is None else min(max_n, len(dataset))
    for i in tqdm(range(n), desc=f"Evaluating {name}"):
        item = dataset[i]
        guess = predict_fn(item)
        truth = float(item.price)
        maes.append(abs(guess - truth))
    mae = float(np.mean(maes))
    med = float(np.median(maes))
    p95 = float(np.percentile(maes, 95))
    print(f"{name} | MAE: ${mae:.2f} | Median: ${med:.2f} | 95th %ile: ${p95:.2f}")
    return np.array(maes)

In [22]:
# Evaluate when the job has status == 'succeeded'
SMOKE = int(os.getenv("SMOKE_EVAL", "8"))  # small, cost-aware

if has_pkl:
    job_info = client.fine_tuning.jobs.retrieve(job.id)
    ft_model = job_info.fine_tuned_model
    if ft_model:
        print("Fine-tuned model:", ft_model, "| mode used:", MODE_USED)
        _ = evaluate_predictor(lambda it: predict_price(it, ft_model, MODE_USED), test_items, max_n=SMOKE, name=f"{MODE_USED} (smoke)")
    else:
        print("Model not ready. Re-run this cell after job status == 'succeeded'.")
else:
    print("No PKL test set detected; upload test.pkl to run evaluation.")

Fine-tuned model: ft:gpt-4o-mini-2024-07-18:personal:pricer-sft-budget:CLEzq8mW | mode used: SFT


Evaluating SFT (smoke):   0%|          | 0/8 [00:00<?, ?it/s]

SFT (smoke) | MAE: $71.46 | Median: $37.40 | 95th %ile: $231.79


## 10) Notes
- If you later increase your quota above `RFT_MIN_START_USD`, set `AVAILABLE_BUDGET_USD` accordingly and re-run to attempt RFT.
- To strictly force SFT (even if you have budget), set the env var `FORCE_MODE=SFT` before running cell **1**.
- To strictly force RFT (and accept creation failure if budget too low), set `FORCE_MODE=RFT` *and* ensure your quota is sufficient.

In [23]:
# Optional force-mode switch (run before section 7 if you want to override auto mode)
_force = os.getenv("FORCE_MODE", "").upper()
if _force in ("RFT","SFT"):
    print(f"NOTE: FORCE_MODE={_force} requested. Set AVAILABLE_BUDGET_USD high enough for RFT.")
else:
    print("No FORCE_MODE override active.")


No FORCE_MODE override active.
